# 11e — Teste de substituição de área (pós-B4.M.4)

**Pré-registro v2.4 + bloco K em construção (v2.5)**

Hipótese a testar (decisão 20/05/2026 após B4.M.4 v2.4 revelar ATT_res_outros < 0 sig 5%):

> Canavieiros certificados RenovaBio expandem área de cana, deslocando outras culturas e/ou ocupando pastagem/vegetação nativa. O ATT negativo em `log1p_res_outros` (SEEG) seria consequência mecânica desta reorganização de uso do solo.

**Dois testes empíricos:**
1. **PAM** (área plantada declarada IBGE) — 4 outcomes: cana, soja, milho, algodão
2. **MapBiomas** (uso do solo observado por satélite) — 6 outcomes: cana, pastagem, vegetação nativa, soja, silvicultura, urbano

**Estimador:** CS-DR idêntico ao 11a/11d. 3 bugs respeitados. **Spec principal: FULL2** (decisão v2.4 com base nos resultados do 11d).

**Total ATTs:** 4 + 6 = 10 outcomes × 4 specs = **40 ATTs**

**Pré-condições no Drive:**
- `pipeline/b4_substituicao_area.py` (módulo deste notebook)
- `pipeline/b4m4_decomposicao.py` v2.4 (para reusar normalize_geocode, etc.)
- `data/interim/panel_canavieiro_main.csv` (842 munis, já tem PAM cana/soja/milho e MB áreas)
- `data/interim/pam_1612_long_2012_2024.parquet` (para join de algodão)
- `data/interim/07_mapbiomas_panel_balanced_2015_2024_CORRIGIDO.csv` (shares MapBiomas)
- `data/raw/psm_baseline/base_psm_integrada_raw.csv`

## Setup

In [15]:
from google.colab import drive
drive.mount("/content/drive")

# Setup portável — resolve a raiz do repositório sem depender do Google Drive.
# Para executar a partir do Drive, defina antes: os.environ["RENOVABIO_BASE_DIR"] = "<caminho>"
# Para executar a partir do Drive, defina antes de rodar esta célula:
import os
import sys
from pathlib import Path

if os.environ.get("RENOVABIO_BASE_DIR"):
    BASE_DIR = Path(os.environ["RENOVABIO_BASE_DIR"]).expanduser().resolve()
else:
    BASE_DIR = Path.cwd().resolve()
    while not (BASE_DIR / "requirements.txt").exists() and BASE_DIR != BASE_DIR.parent:
        BASE_DIR = BASE_DIR.parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
!pip install differences

In [17]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from differences import ATTgt
from pipeline.config import interim, out_pre
from pipeline import b4_substituicao_area as b4s
import importlib; importlib.reload(b4s)

N_BOOT = 999        # producão (SE analitica - bootstrap usado para pointwise CI)
RANDOM_STATE = 42
print(f"setup OK | N_BOOT={N_BOOT} seed={RANDOM_STATE}")
print(f"outcomes PAM: {b4s.OUTCOMES_PAM}")
print(f"outcomes MapBiomas: {b4s.OUTCOMES_MAPB}")

setup OK | N_BOOT=999 seed=42
outcomes PAM: ['log1p_pam_area_cana_t', 'asinh_pam_area_soja_t', 'asinh_pam_area_milho_t', 'asinh_pam_area_algodao_t']
outcomes MapBiomas: ['log1p_share_cana_mapb', 'log1p_share_pastagem_mapb', 'log1p_share_vegetacao_nativa_mapb', 'asinh_share_soja_mapb', 'asinh_share_silvicultura_mapb', 'asinh_share_urbano_infra_mapb']


## Bloco 1 — Carregar painel canônico (idêntico ao 11d Bloco 1)

In [18]:
panel = pd.read_csv(interim("panel_canavieiro_main.csv"), dtype={"geocode": str})
print(f"panel original: {panel.shape}")

COVS_COLIDENTES = ["gini", "densidade_pop", "log_pop", "idhm_renda",
                   "ivs_capital_humano", "ivs_renda_trabalho"]
drop_cols = [c for c in COVS_COLIDENTES if c in panel.columns]
panel = panel.drop(columns=drop_cols)
print(f"apos dropar {len(drop_cols)} colidentes: {panel.shape}")

assert panel["geocode"].nunique() == 842
assert sorted(panel["ano"].unique()) == list(range(2015, 2025))
print("OK 842 munis x 10 anos")

# Sanity: PAM cana/soja/milho ja no painel?
for c in ["pam_area_cana", "pam_area_soja", "pam_area_milho"]:
    assert c in panel.columns, f"{c} ausente"
    print(f"  {c}: media={panel[c].mean():.0f} ha, zeros={(panel[c]==0).sum()}")

panel original: (8420, 139)
apos dropar 6 colidentes: (8420, 133)
OK 842 munis x 10 anos
  pam_area_cana: media=10695 ha, zeros=170
  pam_area_soja: media=10442 ha, zeros=1710
  pam_area_milho: media=7042 ha, zeros=140


## Bloco 2 — Joins: 4 culturas PAM (série temporal) + 6 shares MapBiomas

**v2 (20/05):** descartar as colunas constantes `pam_area_cana/soja/milho` do painel canônico (variância intra-município = 0 → CS-DR retorna ATT=0 SE=NaN) e trazer as 4 culturas da PAM long como série temporal real.

In [19]:
# Join algodao
pam_long = pd.read_parquet(interim("pam_1612_long_2012_2024.parquet"))
print(f"PAM long: {pam_long.shape}")
panel = b4s.join_pam_areas_temporal(panel, pam_long)

# Join MapBiomas
mapb = pd.read_csv(interim("07_mapbiomas_panel_balanced_2015_2024_CORRIGIDO.csv"),
                   dtype={"geocode": str})
print(f"MapBiomas raw: {mapb.shape}")
panel = b4s.join_mapbiomas_shares(panel, mapb)
print(f"painel apos joins: {panel.shape}")

PAM long: (289224, 10)
  dropadas (eram constantes intra-municipio): ['pam_area_cana', 'pam_area_soja', 'pam_area_milho']
  PAM joinada como série temporal:
    pam_area_cana_t       : 832/842 munis variam temporalmente, 208 celulas == 0
    pam_area_soja_t       : 782/842 munis variam temporalmente, 1450 celulas == 0
    pam_area_milho_t      : 836/842 munis variam temporalmente, 202 celulas == 0
    pam_area_algodao_t    : 111/842 munis variam temporalmente, 7865 celulas == 0
MapBiomas raw: (23630, 33)
  mapbiomas: 842 munis no painel canonico, 842 com shares casados
painel apos joins: (8420, 140)


## Bloco 3 — Aplicar transformações (log1p / asinh)

In [20]:
panel = b4s.apply_transformations(panel)

print("\nOutcomes PAM transformados (notna em 8420 esperado):")
for o in b4s.OUTCOMES_PAM:
    nn = panel[o].notna().sum() if o in panel.columns else 0
    print(f"  {o:30s} notna={nn}")

print("\nOutcomes MapBiomas transformados:")
for o in b4s.OUTCOMES_MAPB:
    nn = panel[o].notna().sum() if o in panel.columns else 0
    print(f"  {o:38s} notna={nn}")


Outcomes PAM transformados (notna em 8420 esperado):
  log1p_pam_area_cana_t          notna=8420
  asinh_pam_area_soja_t          notna=8420
  asinh_pam_area_milho_t         notna=8420
  asinh_pam_area_algodao_t       notna=8420

Outcomes MapBiomas transformados:
  log1p_share_cana_mapb                  notna=8420
  log1p_share_pastagem_mapb              notna=8420
  log1p_share_vegetacao_nativa_mapb      notna=8420
  asinh_share_soja_mapb                  notna=8420
  asinh_share_silvicultura_mapb          notna=8420
  asinh_share_urbano_infra_mapb          notna=8420


## Bloco 4 — Reconstruir covariáveis (idêntico ao 11d/11a)

Mesma função `build_covariates_raw`, mesmas 4 specs LEAN/FULL/FULL2/RICH, mesma imputação mediana estadual.

In [21]:
psm_raw = pd.read_csv(
    BASE_DIR / "data/raw/psm_baseline/base_psm_integrada_raw.csv",
    low_memory=False,
)
psm_raw["geocode"] = psm_raw["0_cd_ibge"].astype(str).str.zfill(7)

BIOMA_FIXES = {"Amaz\ufffd\ufffdnia": "Amazônia",
               "Mata Atl\ufffd\ufffdntica": "Mata Atlântica"}
if "14_bioma" in psm_raw.columns:
    psm_raw["14_bioma"] = psm_raw["14_bioma"].replace(BIOMA_FIXES)

muni_id = (panel.groupby("geocode", as_index=False)
           .agg(municipio=("municipio", "first"), uf=("uf", "first"),
                is_treated_ever=("is_treated_ever", "first"),
                g_m=("g_m", "first"), bioma=("bioma", "first")))
muni_id["treated"] = muni_id["is_treated_ever"].astype(int)

df_cs = muni_id.merge(psm_raw, on="geocode", how="inner")
print(f"df_cs: {df_cs.shape}")

df_cs: (842, 172)


In [22]:
def safe_log1p(s, idx):
    s = pd.to_numeric(s, errors="coerce") if s is not None else pd.Series(np.nan, index=idx)
    return np.log1p(s.clip(lower=0))
def safe_div(num, den, idx):
    num = pd.to_numeric(num, errors="coerce") if num is not None else pd.Series(np.nan, index=idx)
    den = pd.to_numeric(den, errors="coerce") if den is not None else pd.Series(np.nan, index=idx)
    return np.where((den.notna()) & (den > 0), num / den, np.nan)
def asn(s, idx):
    return pd.to_numeric(s, errors="coerce") if s is not None else pd.Series(np.nan, index=idx)

def build_covariates_raw(df):
    d = df.copy(); idx = d.index
    d["log_pib_total"]=safe_log1p(d.get("1_pib_total"),idx)
    d["log_pib_pc"]=safe_log1p(d.get("1_pib_percap"),idx)
    d["log_pop"]=safe_log1p(d.get("2_pop_2017_ibge"),idx)
    d["log_area_total"]=safe_log1p(d.get("14_area_total"),idx)
    d["densidade_pop"]=safe_div(d.get("2_pop_2017_ibge"),d.get("14_area_total"),idx)
    d["share_vadc_agro"]=safe_div(d.get("1_vadc_agro"),d.get("1_vadc_bruto"),idx)
    d["share_vadc_ind"]=safe_div(d.get("1_vadc_ind"),d.get("1_vadc_bruto"),idx)
    d["share_vadc_serv"]=safe_div(d.get("1_vadc_serv"),d.get("1_vadc_bruto"),idx)
    d["share_vadc_adm"]=safe_div(d.get("1_vadc_adm"),d.get("1_vadc_bruto"),idx)
    d["share_cana_baseline"]=asn(d.get("3_mb_sharegrp_pre_cana"),idx)
    d["mb_share_soja"]=asn(d.get("3_mb_sharegrp_pre_soja"),idx)
    d["mb_share_pastagem"]=asn(d.get("3_mb_sharegrp_pre_pastagem"),idx)
    d["mb_share_vegetacao_nativa"]=asn(d.get("3_mb_sharegrp_pre_vegetacao_nativa"),idx)
    d["mb_share_urbano"]=asn(d.get("3_mb_sharegrp_pre_urbano_infra"),idx)
    d["log_area_cana"]=safe_log1p(d.get("4_area_colhida_ha_cana"),idx)
    d["log_area_soja"]=safe_log1p(d.get("4_area_colhida_ha_soja"),idx)
    d["log_area_agri_total"]=safe_log1p(d.get("4_area_colhida_ha"),idx)
    d["share_area_cana_agri"]=safe_div(d.get("4_area_colhida_ha_cana"),d.get("4_area_colhida_ha"),idx)
    d["share_est_af"]=safe_div(d.get("5_num_est_af"),d.get("5_num_est_total"),idx)
    d["share_est_mp"]=safe_div(d.get("5_num_est_mp"),d.get("5_num_est_total"),idx)
    d["share_area_af"]=safe_div(d.get("6_area_lav_af"),d.get("6_area_lav_total"),idx)
    d["share_area_mp"]=safe_div(d.get("6_area_lav_mp"),d.get("6_area_lav_total"),idx)
    d["trator_per_est"]=safe_div(d.get("11_num_trator_total"),d.get("5_num_est_total"),idx)
    d["share_est_irrig"]=safe_div(d.get("12_num_est_irrig_total"),d.get("5_num_est_total"),idx)
    d["share_area_irrig"]=safe_div(d.get("12_area_irrig_total"),d.get("6_area_lav_total"),idx)
    d["share_est_fin_total"]=safe_div(d.get("13_num_est_fin_total"),d.get("5_num_est_total"),idx)
    d["share_est_at"]=safe_div(d.get("10_num_est_receb_at"),d.get("5_num_est_total"),idx)
    d["natveg_share_area"]=safe_div(d.get("14_vegetacao_natural"),d.get("14_area_total"),idx)
    d["desmat_share_area"]=safe_div(d.get("14_desmatado"),d.get("14_area_total"),idx)
    d["idhm_renda"]=asn(d.get("17_idhm_renda"),idx); d["idhm_educ"]=asn(d.get("17_idhm_educ"),idx)
    d["ivs_infra"]=asn(d.get("17_ivs_infraestrutura_urbana"),idx); d["gini"]=asn(d.get("17_i_gini"),idx)
    d["idhm_long"]=asn(d.get("17_idhm_long"),idx); d["ivs_capital_humano"]=asn(d.get("17_ivs_capital_humano"),idx)
    d["ivs_renda_trabalho"]=asn(d.get("17_ivs_renda_e_trabalho"),idx)
    d["share_est_at_coop"]=safe_div(d.get("10_num_est_receb_at_coop"),d.get("5_num_est_total"),idx)
    d["share_est_at_gov"]=safe_div(d.get("10_num_est_receb_at_gov"),d.get("5_num_est_total"),idx)
    d["share_fin_invest"]=safe_div(d.get("13_num_est_fin_invest"),d.get("5_num_est_total"),idx)
    d["share_fin_cust"]=safe_div(d.get("13_num_est_fin_cust"),d.get("5_num_est_total"),idx)
    d["share_est_trator"]=safe_div(d.get("11_num_est_trator_total"),d.get("5_num_est_total"),idx)
    d["share_est_irrig_pivo"]=safe_div(d.get("12_num_est_irrig_pivo"),d.get("5_num_est_total"),idx)
    d["pct_est_energia"]=asn(d.get("7_est_com_energia%"),idx)
    d["log_area_milho"]=safe_log1p(d.get("4_area_colhida_ha_milho"),idx)
    d["log_area_alg"]=safe_log1p(d.get("4_area_colhida_ha_alg"),idx)
    d["log_area_cafarab"]=safe_log1p(d.get("4_area_colhida_ha_cafarab"),idx)
    d["log_area_cafcan"]=safe_log1p(d.get("4_area_colhida_ha_cafcan"),idx)
    d["mb_share_cafe"]=asn(d.get("3_mb_sharegrp_pre_cafe"),idx)
    d["mb_share_algodao"]=asn(d.get("3_mb_sharegrp_pre_algodao"),idx)
    d["mb_share_silvicultura"]=asn(d.get("3_mb_sharegrp_pre_silvicultura"),idx)
    d["mb_share_agua"]=asn(d.get("3_mb_sharegrp_pre_agua"),idx)
    d["mb_share_outros"]=asn(d.get("3_mb_sharegrp_pre_outros"),idx)
    d["mb_share_agri_total"]=asn(d.get("3_mb_sharegrp_pre_agricultura_total"),idx)
    d["share_num_est_mp"]=safe_div(d.get("5_num_est_mp"),d.get("5_num_est_total"),idx)
    d["share_est_pec"]=safe_div(d.get("5_num_est_pec_total"),d.get("5_num_est_total"),idx)
    d["share_est_lavperm"]=safe_div(d.get("5_num_est_lavperm_total"),d.get("5_num_est_total"),idx)
    d["share_est_lavtemp"]=safe_div(d.get("5_num_est_lavtemp_total"),d.get("5_num_est_total"),idx)
    d["share_area_lavperm"]=safe_div(d.get("6_area_lavperm_total"),d.get("6_area_lav_total"),idx)
    d["share_area_lavtemp"]=safe_div(d.get("6_area_lavtemp_total"),d.get("6_area_lav_total"),idx)
    d["share_area_pec"]=safe_div(d.get("6_area_pec_total"),d.get("6_area_lav_total"),idx)
    d["share_fin_comer"]=safe_div(d.get("13_num_est_fin_comer"),d.get("5_num_est_total"),idx)
    d["share_est_at_propr"]=safe_div(d.get("10_num_est_receb_at_propr"),d.get("5_num_est_total"),idx)
    d["share_est_at_gov_out"]=safe_div(d.get("10_num_est_receb_at_gov_out"),d.get("5_num_est_total"),idx)
    share_cols = [c for c in d.columns if ("share" in c) or c.startswith("mb_share_")]
    for c in share_cols:
        s = pd.to_numeric(d[c], errors="coerce")
        if not s.dropna().empty and (s.dropna().between(-0.05,1.05).mean() > 0.8):
            d[c] = s.clip(0,1)
    return d

df_cs = build_covariates_raw(df_cs)
print("OK covariaveis reconstruidas")

OK covariaveis reconstruidas


In [23]:
COVS_LEAN = ["log_pib_total","log_pib_pc","log_pop","densidade_pop",
             "share_vadc_agro","share_vadc_ind","share_vadc_serv",
             "share_cana_baseline","mb_share_soja","mb_share_pastagem","mb_share_vegetacao_nativa",
             "log_area_cana","log_area_soja","share_area_cana_agri",
             "share_est_af","share_area_af","trator_per_est","share_est_irrig","share_est_fin_total",
             "ivs_infra","gini"]
COVS_FULL = COVS_LEAN + ["share_vadc_adm","idhm_educ","idhm_renda","idhm_long",
                          "ivs_capital_humano","ivs_renda_trabalho",
                          "share_est_at","share_est_at_coop","share_est_at_gov",
                          "share_fin_invest","share_fin_cust",
                          "share_est_trator","share_est_irrig_pivo","pct_est_energia"]
COVS_FULL2 = [c for c in COVS_FULL if c not in ("share_vadc_agro","share_vadc_ind")]
COVS_RICH_FILTRADAS_OUT = ["mb_share_algodao","log_area_cafcan","mb_share_cafe","mb_share_silvicultura"]
COVS_RICH = COVS_FULL + [c for c in [
    "log_area_milho","log_area_alg","log_area_cafarab","log_area_cafcan",
    "mb_share_cafe","mb_share_algodao","mb_share_silvicultura",
    "mb_share_urbano","mb_share_agua","mb_share_outros","mb_share_agri_total",
    "share_num_est_mp","share_est_pec","share_est_lavperm","share_est_lavtemp",
    "share_area_lavperm","share_area_lavtemp","share_area_pec",
    "share_fin_comer","share_est_at_propr","share_est_at_gov_out",
] if c not in COVS_RICH_FILTRADAS_OUT]
assert len(COVS_RICH) == 52

SPECS = {"LEAN": COVS_LEAN, "FULL": COVS_FULL, "FULL2": COVS_FULL2, "RICH": COVS_RICH}
for name, covs in SPECS.items():
    missing = [c for c in covs if c not in df_cs.columns]
    assert not missing
    print(f"OK {name:6s}: {len(covs)} covs")

all_covs = sorted(set(COVS_RICH))
for c in all_covs:
    if df_cs[c].isna().any():
        df_cs[c] = df_cs.groupby("uf")[c].transform(lambda x: x.fillna(x.median()))
        df_cs[c] = df_cs[c].fillna(df_cs[c].median())
assert df_cs[all_covs].isna().sum().sum() == 0
print(f"OK imputacao, {len(all_covs)} covs unicas")

OK LEAN  : 21 covs
OK FULL  : 35 covs
OK FULL2 : 33 covs
OK RICH  : 52 covs
OK imputacao, 52 covs unicas


## Bloco 5 — Construir painel CS (idêntico ao 11d Bloco 4, Bug 1)

In [24]:
panel_cs = panel.merge(df_cs[["geocode"] + all_covs], on="geocode", how="left")
panel_cs["g_m_cs"] = panel_cs["g_m"]  # Bug 1: NaN = never-treated

n_treated = panel_cs["g_m_cs"].notna().sum() // 10
n_never = panel_cs["g_m_cs"].isna().sum() // 10
print(f"tratados: {n_treated} (esperado 194)")
print(f"nunca-tratados: {n_never} (esperado 648)")
assert n_treated == 194 and n_never == 648
print("OK")

tratados: 194 (esperado 194)
nunca-tratados: 648 (esperado 648)
OK


## Bloco 6 — CS-DR sobre os 4 outcomes PAM × 4 specs (16 ATTs)

Hipótese: ATT_cana_pam > 0 sig + ATT_outras < 0 sig → substituição agrícola direta.

In [25]:
cs_pam = b4s.run_csdr_outcomes(
    panel_cs, outcomes=b4s.OUTCOMES_PAM, specs=SPECS,
    n_boot=N_BOOT, random_state=RANDOM_STATE,
)
cs_pam.to_csv(interim("att_substituicao_pam.csv"), index=False)
print(f"\nOK att_substituicao_pam.csv {cs_pam.shape}")
cs_pam


>>> log1p_pam_area_cana_t
  LEAN   ATT = +0.2682 (SE=0.1547)  [1.9s]
  FULL   ATT = +0.4232 (SE=0.3038)  [5.3s]
  FULL2  ATT = +0.1897 (SE=0.0871)  [3.3s]
  RICH   ATT = +0.4838 (SE=0.4868)  [3.8s]

>>> asinh_pam_area_soja_t
  LEAN   ATT = +0.1735 (SE=0.1290)  [1.6s]
  FULL   ATT = +0.1360 (SE=0.1336)  [2.5s]
  FULL2  ATT = +0.1374 (SE=0.1222)  [6.3s]
  RICH   ATT = +0.1531 (SE=0.1336)  [3.6s]

>>> asinh_pam_area_milho_t
  LEAN   ATT = +0.1498 (SE=0.0853)  [4.7s]
  FULL   ATT = +0.0997 (SE=0.0876)  [8.0s]
  FULL2  ATT = +0.1080 (SE=0.0845)  [2.1s]
  RICH   ATT = +0.0286 (SE=0.0946)  [3.6s]

>>> asinh_pam_area_algodao_t
  LEAN   ATT = +0.2473 (SE=0.1744)  [1.7s]
  FULL   ATT = +0.4814 (SE=0.2546)  [3.0s]
  FULL2  ATT = +0.2664 (SE=0.1421)  [5.7s]
  RICH   ATT = +0.3473 (SE=0.2855)  [3.6s]

OK CS-DR: 16/16 sucessos em 60.7s

OK att_substituicao_pam.csv (16, 8)


,outcome,spec,estimator,ATT,SE,CI_lo,CI_hi,n_munis
0,log1p_pam_area_cana_t,LEAN,CS-DR,0.268237,0.154653,-0.034876,0.571351,842
1,log1p_pam_area_cana_t,FULL,CS-DR,0.423194,0.303821,-0.172285,1.018672,842
2,log1p_pam_area_cana_t,FULL2,CS-DR,0.189720,0.087113,0.018982,0.360459,842
3,log1p_pam_area_cana_t,RICH,CS-DR,0.483784,0.486776,-0.470280,1.437848,842
4,asinh_pam_area_soja_t,LEAN,CS-DR,0.173476,0.129026,-0.079411,0.426362,842
5,asinh_pam_area_soja_t,FULL,CS-DR,0.136012,0.133625,-0.125888,0.397911,842
6,asinh_pam_area_soja_t,FULL2,CS-DR,0.137362,0.122243,-0.102231,0.376954,842
7,asinh_pam_area_soja_t,RICH,CS-DR,0.153141,0.133583,-0.108677,0.414959,842
8,asinh_pam_area_milho_t,LEAN,CS-DR,0.149819,0.085280,-0.017327,0.316965,842
9,asinh_pam_area_milho_t,FULL,CS-DR,0.099707,0.087598,-0.071981,0.271395,842


## Bloco 7 — CS-DR sobre os 6 outcomes MapBiomas × 4 specs (24 ATTs)

Hipótese: ATT_share_cana > 0 + ATT_pastagem|veg_nat < 0 → substituição visível no uso do solo.

**Crítico:** se ATT_vegetacao_nativa < 0 sig → **desmatamento associado a RenovaBio** (achado editorial de alto impacto).

In [26]:
cs_mapb = b4s.run_csdr_outcomes(
    panel_cs, outcomes=b4s.OUTCOMES_MAPB, specs=SPECS,
    n_boot=N_BOOT, random_state=RANDOM_STATE,
)
cs_mapb.to_csv(interim("att_substituicao_mapbiomas.csv"), index=False)
print(f"\nOK att_substituicao_mapbiomas.csv {cs_mapb.shape}")
cs_mapb


>>> log1p_share_cana_mapb
  LEAN   ATT = +0.0037 (SE=0.0011)  [1.7s]
  FULL   ATT = +0.0028 (SE=0.0014)  [2.4s]
  FULL2  ATT = +0.0026 (SE=0.0012)  [4.5s]
  RICH   ATT = +0.0034 (SE=0.0013)  [5.3s]

>>> log1p_share_pastagem_mapb
  LEAN   ATT = -0.0025 (SE=0.0019)  [1.7s]
  FULL   ATT = -0.0027 (SE=0.0023)  [2.5s]
  FULL2  ATT = -0.0024 (SE=0.0016)  [2.1s]
  RICH   ATT = -0.0030 (SE=0.0020)  [7.9s]

>>> log1p_share_vegetacao_nativa_mapb
  LEAN   ATT = -0.0008 (SE=0.0004)  [1.6s]
  FULL   ATT = -0.0008 (SE=0.0004)  [2.6s]
  FULL2  ATT = -0.0003 (SE=0.0004)  [2.1s]
  RICH   ATT = -0.0008 (SE=0.0005)  [8.0s]

>>> asinh_share_soja_mapb
  LEAN   ATT = +0.0010 (SE=0.0015)  [1.6s]
  FULL   ATT = +0.0005 (SE=0.0018)  [2.4s]
  FULL2  ATT = -0.0005 (SE=0.0015)  [2.2s]
  RICH   ATT = -0.0005 (SE=0.0015)  [3.8s]

>>> asinh_share_silvicultura_mapb
  LEAN   ATT = -0.0004 (SE=0.0002)  [5.5s]
  FULL   ATT = -0.0004 (SE=0.0002)  [2.5s]
  FULL2  ATT = -0.0003 (SE=0.0002)  [2.1s]
  RICH   ATT = -0.0003 (

,outcome,spec,estimator,ATT,SE,CI_lo,CI_hi,n_munis
0,log1p_share_cana_mapb,LEAN,CS-DR,0.003678,0.001145,0.001435,0.005922,842
1,log1p_share_cana_mapb,FULL,CS-DR,0.002773,0.001351,0.000124,0.005421,842
2,log1p_share_cana_mapb,FULL2,CS-DR,0.002616,0.001227,0.000212,0.005020,842
3,log1p_share_cana_mapb,RICH,CS-DR,0.003403,0.001330,0.000797,0.006010,842
4,log1p_share_pastagem_mapb,LEAN,CS-DR,-0.002506,0.001890,-0.006209,0.001197,842
5,log1p_share_pastagem_mapb,FULL,CS-DR,-0.002663,0.002298,-0.007167,0.001840,842
6,log1p_share_pastagem_mapb,FULL2,CS-DR,-0.002408,0.001648,-0.005638,0.000822,842
7,log1p_share_pastagem_mapb,RICH,CS-DR,-0.002979,0.001990,-0.006880,0.000922,842
8,log1p_share_vegetacao_nativa_mapb,LEAN,CS-DR,-0.000797,0.000383,-0.001547,-0.000046,842
9,log1p_share_vegetacao_nativa_mapb,FULL,CS-DR,-0.000845,0.000400,-0.001629,-0.000062,842


## Bloco 8 — Diagnóstico de substituição (FULL2 como spec principal)

In [27]:
cfg = b4s.assess_substitution(cs_pam, cs_mapb, spec_principal="FULL2")

print("="*64)
print("DIAGNOSTICO DE SUBSTITUICAO DE AREA (spec FULL2)")
print("="*64)
print(f"\nPAM (area plantada declarada):")
print(f"  cana   > 0 sig 5%: {cfg['pam_cana_pos_sig']}")
print(f"  outras < 0 sig 5%: {cfg['pam_outras_neg_sig']}")
print(f"\nMapBiomas (uso do solo observado):")
print(f"  cana       > 0 sig 5%: {cfg['mapb_cana_pos_sig']}")
print(f"  pastagem   < 0 sig 5%: {cfg['mapb_pastagem_neg_sig']}")
print(f"  veg_nativa < 0 sig 5%: {cfg['mapb_vegnat_neg_sig']}  <-- DESMATAMENTO?")
print()
print(f"  >>> CONFIGURACAO: {cfg['configuracao_substituicao']} <<<")
print(f"  {cfg['narrativa']}")
print()
if cfg['desmatamento_associado']:
    print("  *** ATENCAO ***")
    print("  ATT negativo significante em vegetacao_nativa MapBiomas.")
    print("  Achado editorialmente potente: RenovaBio associado a")
    print("  desmatamento (perda de vegetacao nativa) em canavieiros tratados.")

pd.DataFrame([cfg]).to_csv(interim("att_substituicao_config.csv"), index=False)
print("\nOK att_substituicao_config.csv salvo")

DIAGNOSTICO DE SUBSTITUICAO DE AREA (spec FULL2)

PAM (area plantada declarada):
  cana   > 0 sig 5%: True
  outras < 0 sig 5%: False

MapBiomas (uso do solo observado):
  cana       > 0 sig 5%: True
  pastagem   < 0 sig 5%: False
  veg_nativa < 0 sig 5%: False  <-- DESMATAMENTO?

  >>> CONFIGURACAO: EXPANSAO-SEM-SUB <<<
  Cana expande mas não há contração detectável de outras culturas/usos. Expansão sobre fronteira agrícola nova?


OK att_substituicao_config.csv salvo


## Bloco 9 — Tabela final de significância 5% (todos os 40 ATTs)

In [28]:
Z_CRIT = 1.959963985
print("="*86)
print("TABELA DE SIGNIFICANCIA 5% - 10 outcomes x 4 specs (40 ATTs)")
print("="*86)

todos = pd.concat([cs_pam.assign(grupo="PAM"),
                    cs_mapb.assign(grupo="MapBiomas")], ignore_index=True)
todos["z"] = todos["ATT"] / todos["SE"]
todos["sig_5pct"] = todos["z"].abs() > Z_CRIT
todos["sinal"] = todos["ATT"].apply(lambda x: "+" if x>0 else ("-" if x<0 else "0"))
todos.to_csv(interim("att_substituicao_consolidado.csv"), index=False)

# Resumo por outcome
print(f"\n{'outcome':<38}{'#sig/4':<10}{'sinais':<12}{'veredito'}")
print("-"*86)
for outc in b4s.OUTCOMES_PAM + b4s.OUTCOMES_MAPB:
    sub = todos[todos["outcome"]==outc].sort_values("spec")
    n_sig = int(sub["sig_5pct"].sum())
    sinais = " ".join(sub["sinal"].tolist())
    if n_sig >= 3:
        ver = f"ROBUSTO ({sub['sinal'].mode().iloc[0]})"
    elif n_sig >= 1:
        ver = "PARCIAL"
    else:
        ver = "nulo"
    print(f"{outc:<38}{n_sig}/4{'':<6}{sinais:<12}{ver}")
print()
print("OK att_substituicao_consolidado.csv salvo (para Tabela 5 do paper)")

TABELA DE SIGNIFICANCIA 5% - 10 outcomes x 4 specs (40 ATTs)

outcome                               #sig/4    sinais      veredito
--------------------------------------------------------------------------------------
log1p_pam_area_cana_t                 1/4      + + + +     PARCIAL
asinh_pam_area_soja_t                 0/4      + + + +     nulo
asinh_pam_area_milho_t                0/4      + + + +     nulo
asinh_pam_area_algodao_t              0/4      + + + +     nulo
log1p_share_cana_mapb                 4/4      + + + +     ROBUSTO (+)
log1p_share_pastagem_mapb             0/4      - - - -     nulo
log1p_share_vegetacao_nativa_mapb     2/4      - - - -     PARCIAL
asinh_share_soja_mapb                 0/4      + - + -     nulo
asinh_share_silvicultura_mapb         0/4      - - - -     nulo
asinh_share_urbano_infra_mapb         0/4      - - - -     nulo

OK att_substituicao_consolidado.csv salvo (para Tabela 5 do paper)


## Conclusão B4 substituição

Saídas:
- `att_substituicao_pam.csv` — 4 PAM × 4 specs = 16 ATTs
- `att_substituicao_mapbiomas.csv` — 6 MapBiomas × 4 specs = 24 ATTs
- `att_substituicao_consolidado.csv` — todos os 40 ATTs com sig_5pct, sinal
- `att_substituicao_config.csv` — síntese (Configuração SUB-COMPLETA/PARCIAL/SEM-EXPANSAO)

**Próximo passo:**
- Se SUB-COMPLETA ou SUB-PARCIAL-MAPB com desmatamento: gerar v2.5 incorporando bloco K (substituição confirmada) + revisão da H5.3 (de "ATT ≈ 0" para "ATT < 0 por substituição de área").
- Se SEM-EXPANSAO ou EXPANSAO-SEM-SUB: revisar interpretação — res_outros<0 do B4.M.4 não é por substituição, requer outra explicação.

**Tabela 5 do manuscrito:** consolidado com FULL2 como principal, outras specs em robustez. Inclui flag de desmatamento associado se for o caso.